## Mixup and CutMix Experiments

This notebook mirrors the baseline/dropout/L1/L2 setups but swaps the regularisation for Mixup and CutMix so their impact can be compared under the same model structure.

In [1]:
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mlp.data_providers import EMNISTDataProvider, MixupCutmixDataProvider
from mlp.layers import AffineLayer, ReluLayer, SoftmaxLayer, DropoutLayer
from mlp.errors import CrossEntropySoftmaxError
from mlp.models import MultipleLayerModel
from mlp.initialisers import ConstantInit, GlorotUniformInit
from mlp.learning_rules import AdamLearningRule
from mlp.train_model_and_plot_stats import train_model_and_plot_stats

In [2]:
# Shared experiment settings
seed = 111020
batch_size = 256
learning_rate = 0.0001
num_epochs = 150
stats_interval = 1
input_dim, output_dim, hidden_dim = 784, 47, 256

logger = logging.getLogger()
logger.setLevel(logging.INFO)
logger.handlers = [logging.StreamHandler()]

error = CrossEntropySoftmaxError()


In [3]:
def build_model(rng):
    """3 hidden layer ReLU network with dropout."""
    weights_init = GlorotUniformInit(rng=rng)
    biases_init = ConstantInit(0.)
    layers = [
        AffineLayer(input_dim, hidden_dim, weights_init, biases_init),
        ReluLayer(),
        DropoutLayer(incl_prob=0.8),
        AffineLayer(hidden_dim, hidden_dim, weights_init, biases_init),
        ReluLayer(),
        DropoutLayer(incl_prob=0.8),
        AffineLayer(hidden_dim, hidden_dim, weights_init, biases_init),
        ReluLayer(),
        DropoutLayer(incl_prob=0.8),
        AffineLayer(hidden_dim, output_dim, weights_init, biases_init),
        SoftmaxLayer()
    ]
    return MultipleLayerModel(layers)

def get_data_providers(augmentation=None, mixup_alpha=0.2, cutmix_alpha=1.0,
                       mixup_prob=1.0, cutmix_prob=1.0):
    train_rng = np.random.RandomState(seed)
    valid_rng = np.random.RandomState(seed + 1)
    train_data = EMNISTDataProvider('train', batch_size=batch_size, rng=train_rng, smooth_labels=True)
    valid_data = EMNISTDataProvider('valid', batch_size=batch_size, rng=valid_rng, smooth_labels=False)
    if augmentation is None:
        return train_data, valid_data

    aug_mixup_prob = mixup_prob if augmentation in ['mixup', 'mixup+cutmix'] else 0.0
    aug_cutmix_prob = cutmix_prob if augmentation in ['cutmix', 'mixup+cutmix'] else 0.0
    aug_rng = np.random.RandomState(seed + 2)
    train_data = MixupCutmixDataProvider(
        train_data,
        mixup_alpha=mixup_alpha,
        cutmix_alpha=cutmix_alpha,
        mixup_prob=aug_mixup_prob,
        cutmix_prob=aug_cutmix_prob,
        rng=aug_rng,
    )
    return train_data, valid_data


In [4]:
def run_experiment(label, augmentation=None, mixup_alpha=0.2, cutmix_alpha=1.0,
                   mixup_prob=1.0, cutmix_prob=1.0):
    # Recreate RNG/initialisers each run so results are comparable
    run_rng = np.random.RandomState(seed)
    model = build_model(run_rng)
    learning_rule = AdamLearningRule(learning_rate=learning_rate)
    train_data, valid_data = get_data_providers(
        augmentation=augmentation,
        mixup_alpha=mixup_alpha,
        cutmix_alpha=cutmix_alpha,
        mixup_prob=mixup_prob,
        cutmix_prob=cutmix_prob,
    )
    stats, keys, run_time, fig_1, ax_1, fig_2, ax_2, grad_plot, grad_ax = train_model_and_plot_stats(
        model=model,
        error=error,
        learning_rule=learning_rule,
        train_data=train_data,
        valid_data=valid_data,
        num_epochs=num_epochs,
        stats_interval=stats_interval,
        notebook=True,
    )
    result = {
        'run': label,
        'train_error': stats[-1, keys['error(train)']],
        'val_error': stats[-1, keys['error(valid)']],
        'val_accuracy': stats[-1, keys['acc(valid)']],
        'runtime_s': run_time,
        'fig_error': fig_1,
        'fig_acc': fig_2,
        'fig_grads': grad_plot,
    }
    return result

In [ ]:
experiments = [
        ('baseline_lr1e-4', {'augmentation': None}),
        ('mixup_a0.8_p0.7', {'augmentation': 'mixup', 'mixup_alpha': 0.8, 'mixup_prob': 0.7, 'cutmix_prob':
  0.0}),
        ('cutmix_a1.0_p0.5', {'augmentation': 'cutmix', 'cutmix_alpha': 1.0, 'cutmix_prob': 0.5,
  'mixup_prob': 0.0}),
        ('mix_cut_combo', {'augmentation': 'mixup+cutmix', 'mixup_alpha': 0.4, 'cutmix_alpha': 0.8,
  'mixup_prob': 0.5, 'cutmix_prob': 0.3}),
]



results = []
for label, cfg in experiments:
    print(f"Running {label}...")
    res = run_experiment(label, **cfg)
    results.append(res)

summary = pd.DataFrame([
    {
        'run': r['run'],
        'train_error': r['train_error'],
        'val_error': r['val_error'],
        'val_accuracy': r['val_accuracy'],
        'runtime_s': r['runtime_s'],
    }
    for r in results
])
summary

Running baseline_lr1e-4...
KeysView(NpzFile '/Users/hallaei/UoE/mlp/mlpractical/data/emnist-train.npz' with keys: inputs, targets)
KeysView(NpzFile '/Users/hallaei/UoE/mlp/mlpractical/data/emnist-valid.npz' with keys: inputs, targets)


  0%|          | 0/150 [00:00<?, ?it/s]

  0%|          | 0/390 [00:00<?, ?it/s]

  0%|          | 0/390 [00:00<?, ?it/s]

  0%|          | 0/390 [00:00<?, ?it/s]

...

  0%|          | 0/390 [00:00<?, ?it/s]

  0%|          | 0/390 [00:00<?, ?it/s]

  0%|          | 0/390 [00:00<?, ?it/s]

  0%|          | 0/390 [00:00<?, ?it/s]

  0%|          | 0/390 [00:00<?, ?it/s]

In [ ]:
# Save the summary to a CSV file
summary.to_csv('experiment_summary4.csv', index=False)

Use the plots returned in each result (error, accuracy, gradient flow) to visually compare behaviour. Adjust `mixup_alpha`, `cutmix_alpha`, or probabilities and rerun the loop above for further ablations.